# Stage 7 - End-to-End Evaluation

Cascade the Stage 6 detector into the Stage 5 classifier and measure what the system actually does on continuous video, with nothing handed to it.

### The question this notebook exists to answer

Stage 6 scored **0.781 at +/-5 frames** against a 0.85 target, but **0.845 at +/-8**. I have argued that +/-5 may be too strict - that contact is a racket-ball event, the body moves smoothly through it, and even the source annotations carry ~+/-1 frame of noise.

That argument needs external testing. **The only bar that matters is the one the downstream classifier cares about.**

So: sweep the tolerance, and at each level measure end-to-end accuracy.

- Accuracy **flat out to +/-8** -> +/-8 is the operationally correct bar, detection is fine at 0.845
- Accuracy **degrading sharply past +/-3** -> 0.78 is the correct number and detection is the bottleneck

Either answer is useful. The data decides, not the argument.

### What "end-to-end" means here

A predicted contact is scored correct only if it (a) matches a real contact within tolerance **and** (b) is assigned the right class. Detection errors and classification errors compound, so expect roughly `detection_F1 x classifier_F1` - around **0.62-0.72**, not 0.79.

### Excluded

`test_5` - 27 contacts (1.9%). Its pose stream carries no contact signal at all: mean P(contact) is 0.008 at labelled frames versus 0.006 elsewhere, a ratio of 1.3x where working videos reach 6-21x. Cause undiagnosed; excluded from detection metrics with the reason recorded rather than silently dropped.

GPU required. ~15 min.


## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Setup

Runs standalone: rebuilds both models from the cached artifacts rather than depending on notebook 05 or 06 still being in memory.

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

EXCLUDE   = ["test_5"]          # no contact signal in the pose stream
TOLS      = [1, 2, 3, 5, 8, 12, 20]
SEED      = 42
CLS_EPOCHS = 60
DET_EPOCHS = 40
WINDOW_CLS = 97                 # classifier window, from folds.json
BATCH     = 64

import json, math, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

BASE = Path(BASE); META = BASE/"derived/meta"; CLIP = BASE/"derived/clips"
STREAM = BASE/"derived/pose_stream"; OUT = BASE/"outputs"
(OUT/"metrics").mkdir(parents=True, exist_ok=True)
(OUT/"figures").mkdir(parents=True, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def load(stem):
    p = META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")

strokes = load("strokes")
folds   = json.loads((META/"folds.json").read_text())
V2F, PRE, NF = folds["video2fold"], folds["window"]["pre"], folds["window"]["n_frames"]
CLASSES = ["serve", "attack", "control", "defence"]
C2I = {c: i for i, c in enumerate(CLASSES)}

VIDEOS = [v for v in sorted(strokes.video_id.unique(),
          key=lambda x: (x.split("_")[0], int(x.split("_")[1])))
          if v not in EXCLUDE]
FOLDS = sorted({V2F[v] for v in VIDEOS})

# ground truth: frame -> (class, side) per video
GT = {v: {} for v in VIDEOS}
for r in strokes.itertuples():
    if r.video_id in GT:
        GT[r.video_id][int(r.frame_120)] = (r.shot_class, r.side)

print(f"{len(VIDEOS)} videos ({len(EXCLUDE)} excluded), {len(FOLDS)} folds")
print(f"contacts in scope: {sum(len(g) for g in GT.values())}")

11 videos (1 excluded), 7 folds
contacts in scope: 1430


## 3. Load the pose stream and rebuild features

In [3]:
L_SHO, R_SHO, L_WRI, R_WRI, L_HIP, R_HIP = 5, 6, 9, 10, 11, 12
FLIP = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

def canon(kp, sc, seg, mirror):
    kp = kp.astype(np.float32).copy()
    hip = (kp[:, L_HIP] + kp[:, R_HIP]) / 2
    sho = (kp[:, L_SHO] + kp[:, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)
    scale = np.ones(len(kp), np.float32)
    for s in np.unique(seg):
        m = seg == s; t = torso[m]; t = t[t > 1]
        scale[m] = np.median(t) if len(t) else 1.0
    kp = (kp - hip[:, None, :]) / np.maximum(scale, 1e-3)[:, None, None]
    if mirror:
        kp[..., 0] *= -1
        sc = sc.copy()
        for a, b in FLIP:
            kp[:, [a, b]] = kp[:, [b, a]]; sc[:, [a, b]] = sc[:, [b, a]]
    return kp, sc


S = {}
for v in VIDEOS:
    d = np.load(STREAM/f"{v}.npz", allow_pickle=True)
    fidx, seg = d["frame_idx"], d["seg_id"]
    KP, SC, DET = d["keypoints"], d["scores"], d["detected"]
    ch, kps, vals = [], [], []
    for pi in (0, 1):
        kp, sc = canon(KP[:, pi], SC[:, pi].astype(np.float32), seg, mirror=(pi == 1))
        vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
        vel[np.diff(seg, prepend=seg[0]) != 0] = 0
        ch += [kp.reshape(len(kp), -1), vel.reshape(len(kp), -1),
               (sc * DET[:, pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc >= 0.35) & DET[:, pi:pi+1])
    X = np.nan_to_num(np.concatenate(ch, 1).astype(np.float32))

    pos = np.array([i for i in np.where(d["is_contact"])[0]
                    if int(fidx[i]) in GT[v]], int)
    yc = np.zeros(len(fidx), np.float32); t = np.arange(len(fidx))
    for i in pos:
        lo, hi = max(0, i-8), min(len(fidx), i+9)
        yc[lo:hi] = np.maximum(yc[lo:hi], np.exp(-((t[lo:hi]-i)**2)/8.0))

    cuts = np.where(np.diff(seg) != 0)[0] + 1
    b = np.concatenate([[0], cuts, [len(seg)]])
    S[v] = dict(X=X, yc=yc, pos=pos, fidx=fidx, seg=seg,
                kp=np.stack(kps, 1), val=np.stack(vals, 1),
                spans=[(int(b[i]), int(b[i+1])) for i in range(len(b)-1)],
                fold=V2F[v])
    print(f"  {v}: {len(X):,} frames, {len(pos)} contacts")

C_DET = S[VIDEOS[0]]["X"].shape[1]
print(f"\ndetector input channels: {C_DET}")

  game_1: 20,144 frames, 161 contacts
  game_2: 70,873 frames, 399 contacts
  game_3: 27,911 frames, 153 contacts
  game_4: 24,249 frames, 173 contacts
  game_5: 34,004 frames, 248 contacts
  test_1: 8,027 frames, 84 contacts
  test_2: 2,806 frames, 29 contacts
  test_3: 4,070 frames, 24 contacts
  test_4: 13,251 frames, 71 contacts
  test_6: 5,335 frames, 39 contacts
  test_7: 5,760 frames, 49 contacts

detector input channels: 170


## 4. Models

Both are the architectures already validated: the Stage 6 detector (dilated TCN, full resolution, contact + side heads) and the Stage 5 classifier (dilated TCN, attention pooling, `technique` auxiliary head at 0.20).

In [4]:
class Block(nn.Module):
    def __init__(s, c, d, drop=0.1):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.c2 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.do = nn.Dropout(drop)
    def forward(s, x):
        r = x
        x = s.do(F.gelu(s.n1(s.c1(x))))
        x = s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x + r)

class DetNet(nn.Module):
    def __init__(s, c_in, w=128):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d) for d in (1,2,4,8,16,32,64)])
        s.hc, s.hs = nn.Conv1d(w, 1, 1), nn.Conv1d(w, 1, 1)
    def forward(s, x):
        z = s.blocks(s.stem(x))
        return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s, c):
        super().__init__(); s.score = nn.Conv1d(c, 1, 1)
    def forward(s, x):
        w = torch.softmax(s.score(x), -1)
        return torch.cat([(x*w).sum(-1), x.max(-1).values], -1)

class ClsNet(nn.Module):
    def __init__(s, c_in, w=128, n_tech=8):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d, 0.2) for d in (1,2,4,8,16,32)])
        s.pool = AttnPool(w)
        s.trunk = nn.Sequential(nn.Linear(w*2, 256), nn.GELU(), nn.Dropout(0.3))
        s.shot, s.tech = nn.Linear(256, 4), nn.Linear(256, n_tech)
    def forward(s, x):
        z = s.trunk(s.pool(s.blocks(s.stem(x))))
        return s.shot(z), s.tech(z)

def focal(lg, tg, weight=None, g=2.0):
    m = tg >= 0
    if m.sum() == 0: return lg.sum()*0.0
    lg, tg = lg[m], tg[m]
    ce = F.cross_entropy(lg, tg, weight=weight, reduction="none")
    pt = torch.exp(-F.cross_entropy(lg, tg, reduction="none"))
    return ((1-pt)**g * ce).mean()

print("models defined")

models defined


## 5. Classifier windows from the stream

**The subtle point in this whole notebook.** Stage 5 trained on windows cut around *ground-truth* contact frames. At inference the cascade must cut them around *predicted* frames, which are a couple of frames off.

If the classifier were fed ground-truth-centred windows at evaluation time, the tolerance sweep would measure nothing - the classifier would never see the jitter the detector introduces. So windows are cut from the same pose stream, at whatever frame the detector proposes.

In [5]:
TECHS = ["block","chop","flick","lob","loop","push","serve","smash"]
T2I = {t: i for i, t in enumerate(TECHS)}

def window_at(v, idx, side):
    """97-frame classifier input centred on stream index `idx`."""
    d = S[v]
    lo, hi = idx - PRE, idx - PRE + NF
    sel = np.clip(np.arange(lo, hi), 0, len(d["X"]) - 1)
    pi = 0 if side == "left" else 1
    kp = d["kp"][sel, pi]                       # (T,17,2) already canonical
    val = d["val"][sel, pi].astype(np.float32)
    vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
    x = np.concatenate([kp.reshape(NF, -1), vel.reshape(NF, -1), val,
                        np.zeros((NF, 1), np.float32)], 1)
    return np.nan_to_num(x).T.astype(np.float32)

C_CLS = window_at(VIDEOS[0], S[VIDEOS[0]]["pos"][0], "left").shape[0]
print(f"classifier input: {C_CLS} channels x {NF} frames")

# training set for the classifier: ground-truth windows from the stream
CLS_X, CLS_Y, CLS_T, CLS_V = [], [], [], []
for v in VIDEOS:
    d = S[v]
    for i in d["pos"]:
        f = int(d["fidx"][i]); cls, side = GT[v][f]
        CLS_X.append(window_at(v, i, side)); CLS_Y.append(C2I[cls])
        tech = strokes[(strokes.video_id == v) &
                       (strokes.frame_120 == f)].technique.iloc[0]
        CLS_T.append(T2I.get(tech, -1)); CLS_V.append(v)
CLS_X = np.stack(CLS_X); CLS_Y = np.array(CLS_Y)
CLS_T = np.array(CLS_T); CLS_V = np.array(CLS_V)
CLS_F = np.array([V2F[v] for v in CLS_V])
print(f"classifier training windows: {CLS_X.shape}")

classifier input: 86 channels x 97 frames
classifier training windows: (1430, 86, 97)


## 6. Train both models per fold

In [6]:
def train_det(f, rng):
    tr = [v for v in VIDEOS if S[v]["fold"] != f]
    cat = np.concatenate([S[v]["X"] for v in tr])
    mu, sd = cat.mean(0), cat.std(0) + 1e-6; del cat
    eff = np.mean([(S[v]["yc"] > 0.05).mean() for v in tr])
    pw = torch.tensor((1-eff)/eff, device=dev)
    net = DetNet(C_DET).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = DET_EPOCHS*40
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        xs, ys = [], []
        for _ in range(16):
            v = tr[rng.integers(len(tr))]; d = S[v]
            a, b = d["spans"][rng.integers(len(d["spans"]))]
            n = b - a
            if n <= 512:
                sl = slice(a, b); pad = 512 - n
            else:
                st = a + rng.integers(n-512+1); sl = slice(st, st+512); pad = 0
            x = (d["X"][sl]-mu)/sd; y = d["yc"][sl]
            if pad:
                x = np.pad(x, ((0,pad),(0,0)), mode="edge"); y = np.pad(y, (0,pad))
            xs.append(x.T); ys.append(y)
        x = torch.tensor(np.stack(xs), dtype=torch.float32, device=dev)
        y = torch.tensor(np.stack(ys), dtype=torch.float32, device=dev)
        lc, _ = net(x)
        loss = F.binary_cross_entropy_with_logits(lc, y, pos_weight=pw)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd


def train_cls(f):
    tr = CLS_F != f
    xt = torch.tensor(CLS_X[tr], device=dev)
    mu, sd = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True)+1e-6
    xt = (xt-mu)/sd
    yt = torch.tensor(CLS_Y[tr], device=dev)
    tt = torch.tensor(CLS_T[tr], device=dev)
    cnt = np.bincount(CLS_Y[tr], minlength=4).clip(1)
    cw = torch.tensor(len(CLS_Y[tr])/(4*cnt), dtype=torch.float32, device=dev)
    p = (1.0/cnt)[CLS_Y[tr]]; p = p/p.sum()
    net = ClsNet(C_CLS).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = CLS_EPOCHS*max(1, math.ceil(tr.sum()/BATCH))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        b = np.random.choice(tr.sum(), BATCH, p=p)
        xb = xt[b]
        sh = torch.randint(-6, 7, (len(b),), device=dev)
        ix = (torch.arange(NF, device=dev)[None]+sh[:,None]).clamp(0, NF-1)
        xb = torch.gather(xb, 2, ix[:,None].expand(-1, xb.shape[1], -1))
        xb = xb + torch.randn_like(xb)*0.01
        s_, t_ = net(xb)
        loss = focal(s_, yt[b], cw) + 0.2*focal(t_, tt[b])
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd


def decode(prob, thr, gap=30):
    idx = np.where(prob >= thr)[0]
    if not len(idx): return np.array([], int)
    pk = [i for i in idx if prob[i] == prob[max(0,i-15):i+16].max()]
    pk = sorted(pk, key=lambda i: -prob[i]); kept = []
    for p in pk:
        if all(abs(p-k) >= gap for k in kept): kept.append(p)
    return np.array(sorted(kept), int)


seed_all(SEED); rng = np.random.default_rng(SEED)
PRED = {}
for f in FOLDS:
    va = [v for v in VIDEOS if S[v]["fold"] == f]
    if not va: continue
    dnet, dmu, dsd = train_det(f, rng)
    cnet, cmu, csd = train_cls(f)
    dnet.eval(); cnet.eval()
    for v in va:
        d = S[v]
        prob = np.zeros(len(d["X"]), np.float32)
        sidep = np.zeros(len(d["X"]), np.float32)
        with torch.no_grad():
            for a, b in d["spans"]:
                x = torch.tensor(((d["X"][a:b]-dmu)/dsd).T[None],
                                 dtype=torch.float32, device=dev)
                lc, ls = dnet(x)
                prob[a:b] = torch.sigmoid(lc)[0].cpu().numpy()
                sidep[a:b] = torch.sigmoid(ls)[0].cpu().numpy()
        pk = decode(prob, 0.60)
        if len(pk):
            sides = ["right" if sidep[i] >= 0.5 else "left" for i in pk]
            xb = torch.tensor(np.stack([window_at(v, i, s)
                                        for i, s in zip(pk, sides)]), device=dev)
            with torch.no_grad():
                lo, _ = cnet((xb-cmu)/csd)
                pr = torch.softmax(lo, 1).cpu().numpy()
        else:
            sides, pr = [], np.zeros((0, 4))
        PRED[v] = dict(peaks=pk, sides=sides, proba=pr, prob=prob)
    print(f"  fold {f}: {va} done")
print("\ncascade inference complete")

  fold A: ['game_1'] done
  fold B: ['game_2'] done
  fold C: ['game_3'] done
  fold D: ['game_4'] done
  fold E: ['game_5'] done
  fold F: ['test_1', 'test_4'] done
  fold G: ['test_2', 'test_3', 'test_6', 'test_7'] done

cascade inference complete


## 7. The tolerance sweep

At each tolerance, a prediction counts as **end-to-end correct** only if it matches a real contact within that tolerance *and* carries the right class. Detection-only F1 is reported alongside, so the two error sources stay separable.

In [7]:
def match_pairs(pred, true, tol):
    used, pairs = set(), []
    for k, p in enumerate(pred):
        best, bd = None, tol+1
        for j, t in enumerate(true):
            if j in used: continue
            dist = abs(p-t)
            if dist <= tol and dist < bd: best, bd = j, dist
        if best is not None:
            used.add(best); pairs.append((k, best))
    return pairs

from sklearn.metrics import f1_score

rows = []
for tol in TOLS:
    dTP = dFP = dFN = 0
    yt_all, yp_all = [], []
    for v in VIDEOS:
        d, P = S[v], PRED[v]
        true = d["pos"]
        pairs = match_pairs(P["peaks"], true, tol)
        dTP += len(pairs); dFP += len(P["peaks"])-len(pairs); dFN += len(true)-len(pairs)
        for k, j in pairs:
            f = int(d["fidx"][true[j]])
            yt_all.append(C2I[GT[v][f][0]])
            yp_all.append(int(P["proba"][k].argmax()))
    dp = dTP/max(dTP+dFP,1); dr = dTP/max(dTP+dFN,1)
    df1 = 2*dp*dr/max(dp+dr,1e-9)
    cls_acc = float(np.mean(np.array(yt_all) == np.array(yp_all))) if yt_all else 0.0
    cls_f1 = f1_score(yt_all, yp_all, average="macro", zero_division=0) if yt_all else 0.0
    # end-to-end: correct only if detected AND classified right
    e2e_tp = sum(1 for a, b in zip(yt_all, yp_all) if a == b)
    ep = e2e_tp/max(dTP+dFP,1); er = e2e_tp/max(dTP+dFN,1)
    e2e_f1 = 2*ep*er/max(ep+er,1e-9)
    rows.append(dict(tol=tol, ms=round(tol/120*1000), det_f1=df1,
                     cls_acc=cls_acc, cls_macro_f1=cls_f1, e2e_f1=e2e_f1,
                     n=len(yt_all)))

sweep = pd.DataFrame(rows)
sweep.to_csv(OUT/"metrics/stage7_tolerance_sweep.csv", index=False)
print("=" * 78)
print("TOLERANCE SWEEP")
print("=" * 78)
print(sweep.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

base = float(sweep[sweep.tol == 20].cls_acc.iloc[0])
print("\n" + "=" * 78)
print("DOES THE CLASSIFIER CARE ABOUT DETECTION JITTER?")
print("=" * 78)
print(f"  classifier accuracy on windows centred at the loosest match "
      f"(+/-20): {base:.3f}")
for _, r in sweep.iterrows():
    print(f"    +/-{int(r.tol):<3} ({int(r.ms):>3} ms): acc {r.cls_acc:.3f}   "
          f"{r.cls_acc-base:+.3f}")

a3 = float(sweep[sweep.tol == 3].cls_acc.iloc[0])
a8 = float(sweep[sweep.tol == 8].cls_acc.iloc[0])
drop = a3 - a8
print(f"\n  accuracy at +/-3 vs +/-8: {a3:.3f} vs {a8:.3f}  ({drop:+.3f})")
if abs(drop) < 0.03:
    print("""
  FLAT. Tightening the tolerance from 8 frames to 3 does not make the
  classifier more accurate, so the extra timing precision buys nothing
  downstream. +/-8 (67 ms) is the operationally correct bar, and detection
  at that tolerance is fine.""")
else:
    print(f"""
  NOT FLAT ({drop:+.3f}). Timing precision does matter to the classifier,
  so +/-5 was a reasonable bar after all and detection IS the bottleneck.
  Improving contact localisation is the highest-value next step.""")
print("=" * 78)

TOLERANCE SWEEP
 tol  ms  det_f1  cls_acc  cls_macro_f1  e2e_f1    n
   1   8   0.381    0.510         0.470   0.194  529
   2  17   0.563    0.501         0.470   0.282  782
   3  25   0.677    0.496         0.471   0.336  941
   5  42   0.797    0.497         0.483   0.396 1107
   8  67   0.853    0.491         0.481   0.419 1185
  12 100   0.868    0.492         0.483   0.427 1205
  20 167   0.879    0.493         0.485   0.433 1221

DOES THE CLASSIFIER CARE ABOUT DETECTION JITTER?
  classifier accuracy on windows centred at the loosest match (+/-20): 0.493
    +/-1   (  8 ms): acc 0.510   +0.017
    +/-2   ( 17 ms): acc 0.501   +0.008
    +/-3   ( 25 ms): acc 0.496   +0.003
    +/-5   ( 42 ms): acc 0.497   +0.004
    +/-8   ( 67 ms): acc 0.491   -0.002
    +/-12  (100 ms): acc 0.492   -0.001
    +/-20  (167 ms): acc 0.493   +0.000

  accuracy at +/-3 vs +/-8: 0.496 vs 0.491  (+0.005)

  FLAT. Tightening the tolerance from 8 frames to 3 does not make the
  classifier more accurate, 

## 8. End-to-end result

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

TOL = 8
yt_all, yp_all, per = [], [], []
for v in VIDEOS:
    d, P = S[v], PRED[v]
    true = d["pos"]
    pairs = match_pairs(P["peaks"], true, TOL)
    yt = [C2I[GT[v][int(d["fidx"][true[j]])][0]] for _, j in pairs]
    yp = [int(P["proba"][k].argmax()) for k, _ in pairs]
    yt_all += yt; yp_all += yp
    tp = sum(1 for a, b in zip(yt, yp) if a == b)
    fp = len(P["peaks"]) - tp; fn = len(true) - tp
    p_ = tp/max(tp+fp,1); r_ = tp/max(tp+fn,1)
    per.append(dict(video_id=v, fold=d["fold"], n=len(true),
                    detected=len(pairs), correct=tp,
                    precision=p_, recall=r_,
                    f1=2*p_*r_/max(p_+r_,1e-9)))
pv = pd.DataFrame(per)
pv.to_csv(OUT/"metrics/stage7_per_video.csv", index=False)
print(pv.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n" + classification_report(yt_all, yp_all, target_names=CLASSES,
                                    digits=3, zero_division=0))
cm = confusion_matrix(yt_all, yp_all)
cmdf = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                    columns=[f"pred_{c}" for c in CLASSES])
print(cmdf.to_string())
cmdf.to_csv(OUT/"metrics/stage7_confusion.csv")

TP = pv.correct.sum(); ALL_P = sum(len(PRED[v]["peaks"]) for v in VIDEOS)
ALL_T = sum(len(S[v]["pos"]) for v in VIDEOS)
p_, r_ = TP/max(ALL_P,1), TP/max(ALL_T,1)
e2e = 2*p_*r_/max(p_+r_,1e-9)
det = float(sweep[sweep.tol == TOL].det_f1.iloc[0])
cls = float(sweep[sweep.tol == TOL].cls_macro_f1.iloc[0])
print("\n" + "=" * 78)
print(f"  END-TO-END @ +/-{TOL} frames ({TOL/120*1000:.0f} ms)")
print("=" * 78)
print(f"  detection F1        : {det:.3f}")
print(f"  classification F1   : {cls:.3f}   (isolated: 0.792)")
print(f"  end-to-end F1       : {e2e:.3f}")
print(f"  naive product       : {det*cls:.3f}")
print(f"  target 0.62 - 0.72  : {'PASS' if e2e >= 0.62 else 'BELOW'}")
print("=" * 78)

video_id fold   n  detected  correct  precision  recall    f1
  game_1    A 161       133       78      0.531   0.484 0.506
  game_2    B 399       311      152      0.419   0.381 0.399
  game_3    C 153       135       70      0.429   0.458 0.443
  game_4    D 173       156       69      0.401   0.399 0.400
  game_5    E 248       236       84      0.329   0.339 0.334
  test_1    F  84        65       49      0.653   0.583 0.616
  test_2    G  29        27       16      0.552   0.552 0.552
  test_3    G  24        11        7      0.636   0.292 0.400
  test_4    F  71        62       27      0.391   0.380 0.386
  test_6    G  39        31       17      0.447   0.436 0.442
  test_7    G  49        18       13      0.500   0.265 0.347

              precision    recall  f1-score   support

       serve      0.814     0.465     0.592       245
      attack      0.775     0.542     0.638       552
     control      0.602     0.402     0.482       256
     defence      0.135     0.500     

## 9. Abstention

For a coaching tool an explicit *uncertain* beats a confident error. This is the operating-point choice, not a metric.

In [9]:
conf = np.concatenate([PRED[v]["proba"].max(1) for v in VIDEOS
                       if len(PRED[v]["proba"])])
rows = []
for q in [0.0, 0.05, 0.10, 0.20, 0.30]:
    thr = np.quantile(conf, q) if q > 0 else 0.0
    yt2, yp2, kept = [], [], 0
    for v in VIDEOS:
        d, P = S[v], PRED[v]
        pairs = match_pairs(P["peaks"], d["pos"], TOL)
        for k, j in pairs:
            if P["proba"][k].max() < thr: continue
            kept += 1
            yt2.append(C2I[GT[v][int(d["fidx"][d["pos"][j]])][0]])
            yp2.append(int(P["proba"][k].argmax()))
    acc = float(np.mean(np.array(yt2) == np.array(yp2))) if yt2 else 0
    rows.append(dict(abstain=q, threshold=thr, kept=kept, accuracy=acc))
ab = pd.DataFrame(rows)
print(ab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
ab.to_csv(OUT/"metrics/stage7_abstention.csv", index=False)
print("""
  Reading it: if abstaining on the least-confident 10% lifts accuracy by
  several points, ship with that operating point and surface 'uncertain'
  in the UI rather than guessing.""")

 abstain  threshold  kept  accuracy
   0.000      0.000  1185     0.491
   0.050      0.507  1121     0.502
   0.100      0.557  1057     0.512
   0.200      0.676   938     0.528
   0.300      0.786   821     0.554

  Reading it: if abstaining on the least-confident 10% lifts accuracy by
  several points, ship with that operating point and surface 'uncertain'
  in the UI rather than guessing.


---
## Done

| artifact | purpose |
|---|---|
| `outputs/metrics/stage7_tolerance_sweep.csv` | **the tolerance question, answered** |
| `outputs/metrics/stage7_per_video.csv` | per-video end-to-end |
| `outputs/metrics/stage7_confusion.csv` | end-to-end confusion |
| `outputs/metrics/stage7_abstention.csv` | operating point |

**Read cell 7 first.** Everything else is downstream of whether classifier accuracy is flat across tolerance. That single result decides whether Stage 6's 0.781 was a genuine shortfall or a mis-specified target - and it decides it with data rather than argument.


In [10]:
# =============================================================================
# CELL 10 - STAGE 7 FIX
#
# TWO BUGS in the cells above, both mine.
#
# BUG 1 (severe) - the side head was never trained.
#   train_det computed only the contact loss:
#       lc, _ = net(x)
#       loss = BCE(lc, y, pos_weight=pw)
#   but inference used the side head to choose which player to classify:
#       sides = ["right" if sidep[i] >= 0.5 else "left" for i in pk]
#   So ~half the classifier windows were cut around the WRONG player - the one
#   standing and waiting, not the one striking. An idle player is low-motion,
#   low-amplitude, neutral posture, which reads as `defence`. Hence 488 defence
#   predictions against 132 true, and precision 0.135.
#   It is also a train/test mismatch: CLS_X was built with the GROUND-TRUTH
#   side, so the classifier trained on correct players and was tested on
#   coin-flips.
#
# BUG 2 (minor) - window_at passed np.zeros((NF,1)) where Stage 5 had
#   table_dist, the strongest defence feature in the Stage 4 SHAP ranking.
#
# This cell fixes both and re-runs. Side accuracy is now reported explicitly,
# so the failure mode cannot hide again.
# =============================================================================

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# --- FIX 2: table distance per stream frame ----------------------------------
for v in VIDEOS:
    d = S[v]
    raw = np.load(STREAM/f"{v}.npz", allow_pickle=True)
    tb = raw["table_box"]
    KPr = raw["keypoints"].astype(np.float32)
    hip = (KPr[:, :, L_HIP] + KPr[:, :, R_HIP]) / 2          # (F,2players,2)
    sho = (KPr[:, :, L_SHO] + KPr[:, :, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)               # (F,2)
    td = np.zeros((len(hip), 2), np.float32)
    if tb[2] > tb[0]:
        for pi, edge in ((0, tb[0]), (1, tb[2])):
            t = torso[:, pi]
            med = np.median(t[t > 1]) if (t > 1).any() else 1.0
            td[:, pi] = np.abs(hip[:, pi, 0] - edge) / max(med, 1e-3)
    d["td"] = np.nan_to_num(td)

# --- FIX 1 + 2: side targets, and table_dist in the classifier window --------
for v in VIDEOS:
    d = S[v]
    ys = np.full(len(d["fidx"]), -1.0, np.float32)
    for i in d["pos"]:
        f = int(d["fidx"][i])
        ys[max(0, i-2):i+3] = 0.0 if GT[v][f][1] == "left" else 1.0
    d["ys"] = ys


def window_at(v, idx, side):
    """97-frame classifier input centred on stream index `idx`.
    Channel layout matches Stage 5 exactly: kp(34) vel(34) valid(17) td(1)."""
    d = S[v]
    sel = np.clip(np.arange(idx-PRE, idx-PRE+NF), 0, len(d["X"])-1)
    pi = 0 if side == "left" else 1
    kp = d["kp"][sel, pi]
    val = d["val"][sel, pi].astype(np.float32)
    vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
    td = d["td"][sel, pi][:, None]                    # FIX 2
    x = np.concatenate([kp.reshape(NF, -1), vel.reshape(NF, -1), val, td], 1)
    return np.nan_to_num(x).T.astype(np.float32)


# rebuild the classifier training set with the corrected windows
CLS_X, CLS_Y, CLS_T, CLS_V = [], [], [], []
for v in VIDEOS:
    d = S[v]
    for i in d["pos"]:
        f = int(d["fidx"][i]); cls, side = GT[v][f]
        CLS_X.append(window_at(v, i, side)); CLS_Y.append(C2I[cls])
        tech = strokes[(strokes.video_id == v) &
                       (strokes.frame_120 == f)].technique.iloc[0]
        CLS_T.append(T2I.get(tech, -1)); CLS_V.append(v)
CLS_X = np.stack(CLS_X); CLS_Y = np.array(CLS_Y)
CLS_T = np.array(CLS_T); CLS_V = np.array(CLS_V)
CLS_F = np.array([V2F[v] for v in CLS_V])
C_CLS = CLS_X.shape[1]
print(f"classifier windows rebuilt: {CLS_X.shape}  ({C_CLS} channels, "
      f"table_dist restored)")


# --- FIX 1: detector training WITH the side loss -----------------------------
def train_det2(f, rng):
    tr = [v for v in VIDEOS if S[v]["fold"] != f]
    cat = np.concatenate([S[v]["X"] for v in tr])
    mu, sd = cat.mean(0), cat.std(0) + 1e-6; del cat
    eff = np.mean([(S[v]["yc"] > 0.05).mean() for v in tr])
    pw = torch.tensor((1-eff)/eff, device=dev)

    net = DetNet(C_DET).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = DET_EPOCHS*40
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        xs, ys_, ss = [], [], []
        for _ in range(16):
            v = tr[rng.integers(len(tr))]; d = S[v]
            a, b = d["spans"][rng.integers(len(d["spans"]))]
            n = b - a
            if n <= 512:
                sl = slice(a, b); pad = 512 - n
            else:
                st = a + rng.integers(n-512+1); sl = slice(st, st+512); pad = 0
            x = (d["X"][sl]-mu)/sd; y = d["yc"][sl]; s_ = d["ys"][sl]
            if pad:
                x = np.pad(x, ((0,pad),(0,0)), mode="edge")
                y = np.pad(y, (0,pad)); s_ = np.pad(s_, (0,pad), constant_values=-1)
            xs.append(x.T); ys_.append(y); ss.append(s_)
        x = torch.tensor(np.stack(xs), dtype=torch.float32, device=dev)
        y = torch.tensor(np.stack(ys_), dtype=torch.float32, device=dev)
        s_ = torch.tensor(np.stack(ss), dtype=torch.float32, device=dev)
        lc, ls = net(x)
        loss = F.binary_cross_entropy_with_logits(lc, y, pos_weight=pw)
        m = (s_ >= 0).float()
        if m.sum() > 0:                                   # <-- the missing term
            loss = loss + 0.3*(F.binary_cross_entropy_with_logits(
                ls, s_.clamp(min=0), reduction="none")*m).sum()/m.sum()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd


seed_all(SEED); rng = np.random.default_rng(SEED)
PRED2, side_stats = {}, []
for f in FOLDS:
    va = [v for v in VIDEOS if S[v]["fold"] == f]
    if not va: continue
    dnet, dmu, dsd = train_det2(f, rng)
    cnet, cmu, csd = train_cls(f)
    dnet.eval(); cnet.eval()
    for v in va:
        d = S[v]
        prob = np.zeros(len(d["X"]), np.float32)
        sidep = np.zeros(len(d["X"]), np.float32)
        with torch.no_grad():
            for a, b in d["spans"]:
                x = torch.tensor(((d["X"][a:b]-dmu)/dsd).T[None],
                                 dtype=torch.float32, device=dev)
                lc, ls = dnet(x)
                prob[a:b] = torch.sigmoid(lc)[0].cpu().numpy()
                sidep[a:b] = torch.sigmoid(ls)[0].cpu().numpy()
        pk = decode(prob, 0.60)
        sides = ["right" if sidep[i] >= 0.5 else "left" for i in pk]

        # side accuracy on matched peaks - the check that was missing
        ok = tot = 0
        for k, i in enumerate(pk):
            j = np.abs(d["pos"]-i).argmin() if len(d["pos"]) else None
            if j is None or abs(d["pos"][j]-i) > 8: continue
            tot += 1; ok += int(sides[k] == GT[v][int(d["fidx"][d["pos"][j]])][1])
        side_stats.append(dict(video_id=v, n=tot, side_acc=ok/max(tot,1)))

        if len(pk):
            xb = torch.tensor(np.stack([window_at(v, i, s)
                                        for i, s in zip(pk, sides)]), device=dev)
            with torch.no_grad():
                lo, _ = cnet((xb-cmu)/csd)
                pr = torch.softmax(lo, 1).cpu().numpy()
        else:
            pr = np.zeros((0, 4))
        PRED2[v] = dict(peaks=pk, sides=sides, proba=pr, prob=prob)
    print(f"  fold {f} done")

ss = pd.DataFrame(side_stats)
print("\n" + "=" * 74)
print("SIDE-HEAD ACCURACY  (was untrained; ~0.50 before the fix)")
print("=" * 74)
print(ss.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\n  weighted mean: {(ss.side_acc*ss.n).sum()/ss.n.sum():.3f}")


# --- re-run the tolerance sweep ----------------------------------------------
rows = []
for tol in TOLS:
    dTP = dFP = dFN = 0; yt_all, yp_all = [], []
    for v in VIDEOS:
        d, P = S[v], PRED2[v]
        pairs = match_pairs(P["peaks"], d["pos"], tol)
        dTP += len(pairs); dFP += len(P["peaks"])-len(pairs)
        dFN += len(d["pos"])-len(pairs)
        for k, j in pairs:
            yt_all.append(C2I[GT[v][int(d["fidx"][d["pos"][j]])][0]])
            yp_all.append(int(P["proba"][k].argmax()))
    dp = dTP/max(dTP+dFP,1); dr = dTP/max(dTP+dFN,1)
    df1 = 2*dp*dr/max(dp+dr,1e-9)
    acc = float(np.mean(np.array(yt_all) == np.array(yp_all))) if yt_all else 0
    cf1 = f1_score(yt_all, yp_all, average="macro", zero_division=0) if yt_all else 0
    tp = sum(1 for a, b in zip(yt_all, yp_all) if a == b)
    ep = tp/max(dTP+dFP,1); er = tp/max(dTP+dFN,1)
    rows.append(dict(tol=tol, ms=round(tol/120*1000), det_f1=df1, cls_acc=acc,
                     cls_macro_f1=cf1, e2e_f1=2*ep*er/max(ep+er,1e-9)))
sweep2 = pd.DataFrame(rows)
sweep2.to_csv(OUT/"metrics/stage7_v2_sweep.csv", index=False)

print("\n" + "=" * 74)
print("TOLERANCE SWEEP  (after the fix)")
print("=" * 74)
print(sweep2.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

a3 = float(sweep2[sweep2.tol == 3].cls_acc.iloc[0])
a8 = float(sweep2[sweep2.tol == 8].cls_acc.iloc[0])
print(f"\n  classifier accuracy  +/-3 {a3:.3f}   +/-8 {a8:.3f}   "
      f"diff {a3-a8:+.3f}")
if abs(a3-a8) < 0.03:
    print("""  FLAT - tightening from 8 frames to 3 buys the classifier nothing,
  so +/-8 (67 ms) is the operationally correct bar and detection is fine
  at that tolerance.""")
else:
    print("""  NOT FLAT - timing precision does matter downstream, so +/-5 was a
  fair bar and contact localisation is the bottleneck worth investing in.""")

# --- final -------------------------------------------------------------------
TOL = 8
yt_all, yp_all = [], []
tp_tot = 0
for v in VIDEOS:
    d, P = S[v], PRED2[v]
    pairs = match_pairs(P["peaks"], d["pos"], TOL)
    for k, j in pairs:
        a = C2I[GT[v][int(d["fidx"][d["pos"][j]])][0]]
        b = int(P["proba"][k].argmax())
        yt_all.append(a); yp_all.append(b); tp_tot += int(a == b)

print("\n" + classification_report(yt_all, yp_all, target_names=CLASSES,
                                   digits=3, zero_division=0))
cm = confusion_matrix(yt_all, yp_all)
print(pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                   columns=[f"pred_{c}" for c in CLASSES]).to_string())

ALL_P = sum(len(PRED2[v]["peaks"]) for v in VIDEOS)
ALL_T = sum(len(S[v]["pos"]) for v in VIDEOS)
p_, r_ = tp_tot/max(ALL_P,1), tp_tot/max(ALL_T,1)
e2e = 2*p_*r_/max(p_+r_,1e-9)
det = float(sweep2[sweep2.tol == TOL].det_f1.iloc[0])
cls = float(sweep2[sweep2.tol == TOL].cls_macro_f1.iloc[0])
print("\n" + "=" * 74)
print(f"  END-TO-END @ +/-{TOL} frames ({TOL/120*1000:.0f} ms)")
print("=" * 74)
print(f"  {'':<22}{'before fix':>12}{'after fix':>12}")
print(f"  {'detection F1':<22}{0.853:>12.3f}{det:>12.3f}")
print(f"  {'classification F1':<22}{0.481:>12.3f}{cls:>12.3f}"
      f"   (isolated: 0.792)")
print(f"  {'end-to-end F1':<22}{0.419:>12.3f}{e2e:>12.3f}")
print(f"\n  target 0.62 - 0.72  ->  {'PASS' if e2e >= 0.62 else 'BELOW'}")
print("=" * 74)

classifier windows rebuilt: (1430, 86, 97)  (86 channels, table_dist restored)
  fold A done
  fold B done
  fold C done
  fold D done
  fold E done
  fold F done
  fold G done

SIDE-HEAD ACCURACY  (was untrained; ~0.50 before the fix)
video_id   n  side_acc
  game_1 139     1.000
  game_2 326     0.994
  game_3 135     1.000
  game_4 155     0.974
  game_5 237     1.000
  test_1  79     1.000
  test_4  63     0.984
  test_2  27     1.000
  test_3  14     1.000
  test_6  31     1.000
  test_7  27     1.000

  weighted mean: 0.994

TOLERANCE SWEEP  (after the fix)
 tol  ms  det_f1  cls_acc  cls_macro_f1  e2e_f1
   1   8   0.416    0.816         0.758   0.340
   2  17   0.603    0.809         0.755   0.487
   3  25   0.713    0.805         0.758   0.574
   5  42   0.829    0.794         0.760   0.658
   8  67   0.881    0.792         0.762   0.698
  12 100   0.892    0.790         0.762   0.705
  20 167   0.901    0.790         0.762   0.712

  classifier accuracy  +/-3 0.805   +/-8 0.79